# TEST — Create Silver Tables
Standalone test version of `setup/create_silver_tables.ipynb`.  
All parameters are hardcoded below — just edit and **Run All**.

Safe to re-run — uses `mode('ignore')` so existing tables are never touched.

In [ ]:
# ============================================================
# TEST PARAMETERS — edit these to match your test scenario
# ============================================================
WORKSPACE = "dev-fabric-data"     # your Fabric workspace name
LAKEHOUSE = "fabric_lakehouse"    # your Lakehouse name

In [ ]:
import sys

from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField,
    StringType, DoubleType, DateType,
)

sys.path.insert(0, "/lakehouse/default/Files/shared")
import shared_functions as gf

spark = SparkSession.builder.getOrCreate()

print(f"[TEST setup/create_silver_tables] workspace={WORKSPACE} lakehouse={LAKEHOUSE}")

### Helper

In [ ]:
def create_table(table_name: str, schema: StructType) -> None:
    """Write an empty DataFrame with the given schema as a Delta table.
    mode('ignore') means: create if not exists, skip if it already exists.
    """
    path = gf.onelake_abfss(WORKSPACE, LAKEHOUSE, f"Tables/silver/{table_name}")
    (
        spark.createDataFrame([], schema)
        .write
        .format("delta")
        .mode("ignore")
        .save(path)
    )
    print(f"  OK  {table_name}")

### Silver 1 — Staging tables

In [ ]:
print("[1/2] Silver 1 staging tables")

meta = [
    StructField("_source_name", StringType(), True),
    StructField("_source_file", StringType(), True),
    StructField("_run_id",      StringType(), True),
]

create_table("staging_fedex_rate_card", StructType([
    StructField("effective_date",     DateType(),   True),
    StructField("expiry_date",        DateType(),   True),
    StructField("base_rate",          DoubleType(), True),
    StructField("fuel_surcharge_pct", DoubleType(), True),
] + meta))

### Silver 2 — Combined tables

In [ ]:
print("[2/2] Silver 2 combined tables")

create_table("silver_carrier_rate_card", StructType([
    StructField("carrier",             StringType(),    True),
    StructField("service_type",        StringType(),    True),
    StructField("zone",                StringType(),    True),
    StructField("weight_break",        DoubleType(),    True),
    StructField("base_rate",           DoubleType(),    True),
    StructField("fuel_surcharge_pct",  DoubleType(),    True),
    StructField("effective_date",      DateType(),      True),
    StructField("expiry_date",         DateType(),      True),
    StructField("_source_name",        StringType(),    True),
    StructField("_source_file",        StringType(),    True),
    StructField("_run_id",             StringType(),    True),
]))

print("\nAll silver tables created (or already existed).")